In [46]:
import fastf1
import pandas as pd
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

## Load 2023 Japan Grand Prix for Understanding

In [ ]:
# Enable caching to speed up data loading (optional but recommended)
# Cache directory will be created automatically in the scripts directory
if '__file__' in globals():
    # Running as a script
    file_path = os.path.abspath(__file__)
    scripts_dir = os.path.dirname(file_path)
    cache_dir = os.path.join(scripts_dir, 'cache')
else:
    # Running in IPython/interactive mode - use current working directory
    cache_dir = os.path.join(os.getcwd(), 'cache')

# Create cache directory if it doesn't exist
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

# Load a session (example: 2023 Bahrain Grand Prix, Race)
year = 2023
gp = 'Japan'
session_type = 'R'  # R = Race, Q = Qualifying, FP1/FP2/FP3 = Practice

session = fastf1.get_session(year, gp, session_type)
session.load()  # Load all available data for this session

# gathering event metadata
race_type = session.name
print(f"race type: {race_type}")
race_date = session.date # event date
print(f"race date: {race_date}")
race_round = session.event.RoundNumber
print(f"race round: {race_round}")
country = session.event.Country
print(f"country: {country}")
location = session.event.Location
print(f"location: {location}")


core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 00:00.076000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '44', '55', '63', '14', '31', '10', '40', '2

race type: Race
race date: 2023-09-24 05:00:00
race round: 16
country: Japan
location: Suzuka


In [50]:
session.event

RoundNumber                                                  16
Country                                                   Japan
Location                                                 Suzuka
OfficialEventName    FORMULA 1 LENOVO JAPANESE GRAND PRIX 2023 
EventDate                                   2023-09-24 00:00:00
EventName                                   Japanese Grand Prix
EventFormat                                        conventional
Session1                                             Practice 1
Session1Date                          2023-09-22 11:30:00+09:00
Session1DateUtc                             2023-09-22 02:30:00
Session2                                             Practice 2
Session2Date                          2023-09-22 15:00:00+09:00
Session2DateUtc                             2023-09-22 06:00:00
Session3                                             Practice 3
Session3Date                          2023-09-23 11:30:00+09:00
Session3DateUtc                         

In [51]:
session.start_time

AttributeError: 'Session' object has no attribute 'start_time'

## Examine the Laps Object

### Dataframe Description

each row is a lap for a given driver

### Describe Columns

#### Basic Lap Info

- Time: Timestamp when the lap was completed (relative to session start)
- LapStartTime: start time of the lap relative to session start
- Driver: Driver abbreviation (e.g., "VER", "HAM")
- DriverNumber: Driver's permanent number
- LapTime: Total time to complete the lap (timedelta)
- LapNumber: Sequential lap number for that driver
- Stint: Stint number (continuous period between pit stops)
- Team: the team

#### Pit Stop Info

- PitOutTime: Timestamp when driver exited the pit lane (NaT if no pit stop)
- PitInTime: Timestamp when driver entered the pit lane (NaT if no pit stop)
  
#### Sector Times Info

- Sector1Time: Time for sector 1
- Sector2Time: Time for sector 2
- Sector3Time: Time for sector 3
- Sector1SessionTime: Timestamp when sector 1 was completed
- Sector2SessionTime: Timestamp when sector 2 was completed
- Sector3SessionTime: Timestamp when sector 3 was completed

#### Speed Measurement Info

- SpeedI1: Speed at intermediate timing point 1
- SpeedI2: Speed at intermediate timing point 2
- SpeedFL: Speed at the finish line
- SpeedST: Speed at the start line

#### Performance Flag Info

- IsPersonalBest: Boolean indicating if this is the driver's personal best lap

#### Tire Info

- Compound: Tire compound used (e.g., "SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET")
- TyreLife: Number of laps on the current set of tires
- FreshTyre: Boolean indicating if tires were new at the start of the lap

#### Race Status Info

- TrackStatus: Numeric code indicating track conditions (e.g., 1=clear, 4=yellow flag, 41=SC, etc.)
- Position: Driver's race position at the end of the lap

#### Data Quality Flags Info

- Deleted: Boolean indicating if the lap was deleted (e.g., for track limits)
- DeletedReason: Reason why the lap was deleted (if applicable)
- FastF1Generated: Boolean indicating if the lap was generated by FastF1 (may be less accurate)
- IsAccurate: Boolean indicating if the lap data is considered accurate

### Edge-Case Study

In [52]:
max_laps_df = laps_df.loc[laps_df["Driver"] == "VER", :]
max_lap1_df = max_laps_df.loc[max_laps_df["LapNumber"] == 1.0, :]
max_laps_df.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:05:04.229000,VER,1,0 days 00:02:00.179000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:49.276000,...,True,Red Bull Racing,0 days 01:03:03.844000,2023-09-24 05:04:05.253,124,1.0,False,,False,False
1,0 days 01:07:48.591000,VER,1,NaT,2.0,1.0,NaT,NaT,0 days 00:01:05.251000,0 days 00:01:08.600000,...,True,Red Bull Racing,0 days 01:05:04.229000,2023-09-24 05:06:05.638,4,1.0,False,,False,False
2,0 days 01:10:33.919000,VER,1,NaT,3.0,1.0,NaT,NaT,0 days 00:01:08.931000,0 days 00:01:08.363000,...,True,Red Bull Racing,0 days 01:07:48.591000,2023-09-24 05:08:50.000,4,1.0,False,,False,False
3,0 days 01:13:29.577000,VER,1,NaT,4.0,1.0,NaT,NaT,0 days 00:01:00.867000,0 days 00:01:06.499000,...,True,Red Bull Racing,0 days 01:10:33.919000,2023-09-24 05:11:35.328,41,1.0,False,,False,False
4,0 days 01:15:06.325000,VER,1,0 days 00:01:36.748000,5.0,1.0,NaT,NaT,0 days 00:00:34.900000,0 days 00:00:42.960000,...,True,Red Bull Racing,0 days 01:13:29.577000,2023-09-24 05:14:30.986,1,1.0,False,,False,True


In [53]:
# Simply transpose the single row
print(max_lap1_df.T)

                                             0
Time                    0 days 01:05:04.229000
Driver                                     VER
DriverNumber                                 1
LapTime                 0 days 00:02:00.179000
LapNumber                                  1.0
Stint                                      1.0
PitOutTime                                 NaT
PitInTime                                  NaT
Sector1Time                                NaT
Sector2Time             0 days 00:00:49.276000
Sector3Time             0 days 00:00:35.061000
Sector1SessionTime                         NaT
Sector2SessionTime      0 days 01:04:29.265000
Sector3SessionTime      0 days 01:05:04.405000
SpeedI1                                  278.0
SpeedI2                                  124.0
SpeedFL                                  197.0
SpeedST                                  122.0
IsPersonalBest                           False
Compound                                MEDIUM
TyreLife     

### Quality Check

#### Mantra

Your goal is to:
- Discover what the data claims to be, and where reality violates those claims.

Think of QC as:
- Asking questions the data must answer
- Looking for violations of basic invariants

In [ ]:
max_lap1_df.loc[:, "Sector1Time"] + max_lap1_df.loc[:, "Sector2Time"] + max_lap1_df.loc[:,"Sector3Time"]

0   NaT
dtype: timedelta64[ns]

In [25]:
max_lap1_df.loc[:, "Sector1Time"]

0   NaT
Name: Sector1Time, dtype: timedelta64[ns]

In [26]:
max_lap1_df.loc[:, "Sector2Time"]

0   0 days 00:00:49.276000
Name: Sector2Time, dtype: timedelta64[ns]

In [28]:
max_lap1_df.loc[:, "Sector2SessionTime"] + max_lap1_df.loc[:,"Sector3SessionTime"]

0   0 days 02:09:33.670000
dtype: timedelta64[ns]

looks like session times don't match up to lap time. that's fine. its not a big deal. just avoid.

In [29]:
max_lap1_df.loc[:, "Compound"]

0    MEDIUM
Name: Compound, dtype: object

In [30]:
max_lap1_df["Compound"]

0    MEDIUM
Name: Compound, dtype: object

In [32]:
laps_df.loc[:, "Compound"].unique()

array(['MEDIUM', 'HARD', 'SOFT'], dtype=object)

In [33]:
laps_df.loc[:, "LapTime"]

0     0 days 00:02:00.179000
1                        NaT
2                        NaT
3                        NaT
4     0 days 00:01:36.748000
               ...          
875   0 days 00:02:20.615000
876                      NaT
877   0 days 00:02:02.755000
878   0 days 00:01:46.852000
879                      NaT
Name: LapTime, Length: 880, dtype: timedelta64[ns]

In [ ]:
# so laptime has NaT values, meaning its not an actual race?

In [45]:
laptime_nat_mask = pd.isna(laps_df["LapTime"])
laps_df.loc[laptime_nat_mask,:]

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
1,0 days 01:07:48.591000,VER,1,NaT,2.0,1.0,NaT,NaT,0 days 00:01:05.251000,0 days 00:01:08.600000,...,True,Red Bull Racing,0 days 01:05:04.229000,2023-09-24 05:06:05.638,4,1.0,False,,False,False
2,0 days 01:10:33.919000,VER,1,NaT,3.0,1.0,NaT,NaT,0 days 00:01:08.931000,0 days 00:01:08.363000,...,True,Red Bull Racing,0 days 01:07:48.591000,2023-09-24 05:08:50.000,4,1.0,False,,False,False
3,0 days 01:13:29.577000,VER,1,NaT,4.0,1.0,NaT,NaT,0 days 00:01:00.867000,0 days 00:01:06.499000,...,True,Red Bull Racing,0 days 01:10:33.919000,2023-09-24 05:11:35.328,41,1.0,False,,False,False
54,0 days 01:07:50.842000,NOR,4,NaT,2.0,1.0,NaT,NaT,0 days 00:01:04.471000,0 days 00:01:08.019000,...,True,McLaren,0 days 01:05:07.414000,2023-09-24 05:06:08.823,4,2.0,False,,False,False
55,0 days 01:10:35.862000,NOR,4,NaT,3.0,1.0,NaT,NaT,0 days 00:01:07.602000,0 days 00:01:08.917000,...,True,McLaren,0 days 01:07:50.842000,2023-09-24 05:08:52.251,4,2.0,False,,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
870,0 days 01:30:50.927000,PER,11,NaT,13.0,3.0,0 days 01:28:02.755000,0 days 01:30:49.377000,NaT,0 days 00:01:05.514000,...,False,Red Bull Racing,0 days 01:27:22.534000,2023-09-24 05:28:23.943,167,19.0,False,,False,False
871,0 days 02:13:59.369000,PER,11,NaT,14.0,4.0,0 days 02:12:15.286000,0 days 02:13:58.191000,NaT,0 days 00:00:43.773000,...,False,Red Bull Racing,0 days 01:30:50.927000,2023-09-24 05:31:52.336,1,19.0,False,,False,False
874,0 days 01:08:46.269000,BOT,77,NaT,2.0,2.0,0 days 01:06:53.153000,NaT,0 days 00:01:36.832000,0 days 00:00:48.868000,...,True,Alfa Romeo,0 days 01:05:58.728000,2023-09-24 05:07:00.137,4,20.0,False,,False,False
876,0 days 01:13:39.687000,BOT,77,NaT,4.0,2.0,NaT,NaT,0 days 00:00:53.577000,0 days 00:01:01.454000,...,True,Alfa Romeo,0 days 01:11:06.884000,2023-09-24 05:12:08.293,41,20.0,False,,False,False


In [ ]:
# so laptime is not dependable, since it contains NaT (Not A Time) values. Then use Time column instead.
# it's okay if the lap is void because there is a safety car, etc. you should need a consistent data point